In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
import os

In [3]:
from data_scripts.data_import import BaseballVideos

In [4]:

def moving_collate(batch):
    """
    batch: list of (img, target) from Dataset.__getitem__
    img:    torch.Tensor [C,H,W], dtype uint8
    target: dict with key "moving" as list[bool]
    """
    imgs, targets = zip(*batch)  # tuples of length B

    # Convert images to float32 in [0,1] and stack
    imgs = torch.stack(
        [img.to(torch.float32) / 255.0 for img in imgs],
        dim=0
    )  # [B, C, H, W]

    # Binary label: 1.0 if ANY moving box, else 0.0
    labels = []
    for t in targets:
        movings = t["moving"]  # list[bool]
        if len(movings) == 0:
            labels.append(0.0)
        else:
            labels.append(1.0 if any(movings) else 0.0)

    labels = torch.tensor(labels, dtype=torch.float32)  # [B]

    return imgs, labels


In [5]:
directory = os.getcwd()

In [6]:
validation_directory = os.path.join(directory,"validation")
validation_directory

'/Users/josephcoldanghise/Desktop/Baseball Project/baseball_project/validation'

In [7]:
os.chdir(validation_directory)

In [8]:
os.getcwd()

'/Users/josephcoldanghise/Desktop/Baseball Project/baseball_project/validation'

In [9]:
val_dataset = BaseballVideos()

In [10]:
#dataset = BaseballVideos()
print("Total frames:", len(val_dataset))

# Look at a single sample
img, target = val_dataset[0]
print("img shape:", img.shape, "dtype:", img.dtype)
print("target keys:", target.keys())
print("moving list:", target["moving"])

# Try collate on a small manual batch
batch = [val_dataset[i] for i in range(4)]
imgs, labels = moving_collate(batch)
print("batch imgs:", imgs.shape, imgs.dtype)   # expect [4, 3, H, W], float32
print("batch labels:", labels)                # e.g. tensor([0., 1., 1., 0.])


Total frames: 140
img shape: torch.Size([3, 3840, 2160]) dtype: torch.uint8
target keys: dict_keys(['boxes', 'labels', 'area', 'moving'])
moving list: [False]
batch imgs: torch.Size([4, 3, 3840, 2160]) torch.float32
batch labels: tensor([0., 0., 0., 0.])


In [11]:
val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,       # 0 keeps things simple & predictable
    collate_fn=moving_collate
)

In [12]:
# -----------------------------
# 1) Basic config (keep memory low)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4        # small batch -> less GPU/CPU RAM
NUM_EPOCHS = 5
LR = 1e-3

In [13]:
# -----------------------------
# 3) Tiny CNN for "moving vs not moving"
# -----------------------------

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # [B, 64, 1, 1]
        )
        self.fc = nn.Linear(64, 1)    # output: logit for "moving"

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)     # [B, 64]
        x = self.fc(x)                # [B, 1]
        return x.squeeze(1)           # [B]

model = SmallCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [14]:
os.chdir(directory)

In [15]:
# For reloading
import torch
#from train_moving import SmallCNN  # or redefine the same class here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

checkpoint = torch.load("pytorch files/moving_classifier4.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]

model.eval()  # ready for inference
print(f"Restored model from epoch {start_epoch}")


Restored model from epoch 5


In [16]:
print(checkpoint.keys())

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict'])


In [17]:
for name, p in model.named_parameters():
    print(name, p.data.norm().item())


features.0.weight 2.4057366847991943
features.0.bias 0.4875938296318054
features.2.weight 5.158190727233887
features.2.bias 0.2821491062641144
features.4.weight 10.65842342376709
features.4.bias 0.3556155264377594
fc.weight 0.7181165814399719
fc.bias 0.15469424426555634


In [18]:
img, target = val_dataset[41]   # ANY index
img = img.to(torch.float32) / 255.0
img = img.unsqueeze(0).to(device)  # → [1, 3, H, W]

with torch.no_grad():
    logit = model(img)           # model outputs a logit
    prob = torch.sigmoid(logit)  # convert to probability
    pred = (prob >= 0.5).float()

print("logit:", logit.item())
print("probability of moving:", prob.item())
print("predicted label:", pred.item())



logit: -1.9471980333328247
probability of moving: 0.12485920637845993
predicted label: 0.0


In [19]:
img, target = val_dataset[41] 

In [20]:
type(img)

torch.Tensor

dict

In [64]:
@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()

    all_probs = []
    all_targets = []

    for images, targets in dataloader:
        images = images.to(device)
        moving = targets.to(device).float()   # ✅ tensor of shape [B]

        logits = model(images)                # [B]
        probs = torch.sigmoid(logits)         # [B]

        all_probs.append(probs.cpu())
        all_targets.append(moving.cpu())

    all_probs = torch.cat(all_probs)
    all_targets = torch.cat(all_targets)

    # Inspect class balance / predictions
    preds = (all_probs > 0.5).int()
    print("Unique targets:", torch.unique(all_targets, return_counts=True))
    print("Unique preds:", torch.unique(preds, return_counts=True))

    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    y_true = all_targets.numpy()
    y_pred = preds.numpy()
    y_prob = all_probs.numpy()
    

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        # "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        # "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
    }
    
    values = {
        "y_values": y_true,
        "predictions": y_pred,
        "probabilites": y_prob,
    }
    
    return metrics, values


In [65]:
metrics = evaluate_model(model, val_loader, device)
print(metrics)

Unique targets: (tensor([0., 1.]), tensor([86, 54]))
Unique preds: (tensor([0], dtype=torch.int32), tensor([140]))
({'accuracy': 0.6142857142857143, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, {'y_values': array([1., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0.,
       1., 0., 1., 1., 1., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0.,
       0., 1., 0., 0., 0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 0., 1., 0.,
       0., 1., 1., 0., 0., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 1., 1., 1., 0., 1., 1., 0., 1., 0., 1., 1., 0.,
       0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 1., 0., 1., 0., 1., 0., 0.,
       0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 1., 0.,
       1., 0., 0., 1.], dtype=float32), 'predictions': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

/Users/josephcoldanghise/.pyenv/versions/baseball_project/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [1]:
metrics = evaluate_model(model, val_loader, device)
print(metrics)

NameError: name 'evaluate_model' is not defined

In [63]:
len(metrics[1]['probabilites'])

140